In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [3]:
# Load the CSV file
df = pd.read_csv(r"C:\DTT ML 2026\04-evaluation\course_lead_scoring_2026.csv")

In [4]:
# Check that the dataset loaded properly and view the first few rows
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
# Feature categorization
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [6]:
# Imputation
for col in categorical:
    df[col] = df[col].fillna('NA')

for col in numerical:
    df[col] = df[col].fillna(0.0)

In [7]:
# Exact 60% / 20% / 20% split with random_state=1
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print("Train shape:", df_train.shape)
print("Val shape:  ", df_val.shape)
print("Test shape: ", df_test.shape)

Train shape: (3000, 8)
Val shape:   (1000, 8)
Test shape:  (1000, 8)


In [ ]:
# Question 1: ROC AUC Feature Importance
# Evaluate the ROC AUC score for each numerical variable individually against y_train. If a score falls below 0.5, invert the feature sign

auc_scores = {}

for col in numerical:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    auc_scores[col] = auc
    print(f"  {col:<26}: {auc:.4f}")

best_num_feat = max(auc_scores, key=auc_scores.get)

print(f"\nHighest AUC numerical variable: {best_num_feat} ({auc_scores[best_num_feat]:.4f})")

  annual_income             : 0.6082
  number_of_courses_viewed  : 0.7230
  interaction_count         : 0.7649
  lead_score                : 0.7882

Highest AUC numerical variable: lead_score (0.7882)


In [11]:
# Question 2: Model Training & Validation AUC
# One-hot encode all features using DictVectorizer and train LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred_val = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_pred_val)

print(f"Validation AUC: {val_auc:.4f}")
print(f"Rounded to 3 decimal digits: {val_auc:.3f}")

Validation AUC: 0.7318
Rounded to 3 decimal digits: 0.732


In [12]:
thresholds = np.linspace(0.0, 1.0, 101)
scores = []

for t in thresholds:
    actual_pos = (y_val == 1)
    actual_neg = (y_val == 0)
    
    pred_pos = (y_pred_val >= t)
    pred_neg = (y_pred_val < t)
    
    tp = (pred_pos & actual_pos).sum()
    tn = (pred_neg & actual_neg).sum()
    fp = (pred_pos & actual_neg).sum()
    fn = (pred_neg & actual_pos).sum()
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    
    scores.append((t, p, r, f1))

df_metrics = pd.DataFrame(scores, columns=['threshold', 'precision', 'recall', 'f1'])

In [14]:
# Question 3: Precision / Recall Intersection

valid_p_r = df_metrics[(df_metrics['precision'] > 0) & (df_metrics['recall'] > 0)].copy()
valid_p_r['abs_diff'] = (valid_p_r['precision'] - valid_p_r['recall']).abs()
intersect_row = valid_p_r.sort_values(by='abs_diff').iloc[0]

print(f"Intersection Threshold: {intersect_row['threshold']:.2f}")
print(f"Precision: {intersect_row['precision']:.4f} | Recall: {intersect_row['recall']:.4f}")

Intersection Threshold: 0.63
Precision: 0.7328 | Recall: 0.7253


In [15]:
# Question 4: Maximum F1 Threshold

best_f1_row = df_metrics.sort_values(by='f1', ascending=False).iloc[0]

print(f"Maximal F1 Threshold: {best_f1_row['threshold']:.2f} (F1 = {best_f1_row['f1']:.4f})")

Maximal F1 Threshold: 0.41 (F1 = 0.7576)


In [17]:
from sklearn.model_selection import KFold

# Helper functions for cross-validation on df_full_train
def train_model(df_tr, y_tr, C=1.0):
    dicts = df_tr[categorical + numerical].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(dicts)
    
    clf = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    clf.fit(X, y_tr)
    return dv, clf

def predict_model(df_va, dv, clf):
    dicts = df_va[categorical + numerical].to_dict(orient='records')
    X = dv.transform(dicts)
    return clf.predict_proba(X)[:, 1]

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
y_full = df_full_train['converted'].values

In [18]:
# Question 5: Standard Deviation across 5 Folds
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(df_full_train)):
    df_tr = df_full_train.iloc[train_idx]
    df_va = df_full_train.iloc[val_idx]
    
    dv, clf = train_model(df_tr, y_full[train_idx], C=1.0)
    y_pred = predict_model(df_va, dv, clf)
    
    auc = roc_auc_score(y_full[val_idx], y_pred)
    fold_aucs.append(auc)

std_auc = np.std(fold_aucs)
print(f"Mean AUC: {np.mean(fold_aucs):.4f}")
print(f"Standard Deviation: {std_auc:.4f} | Round (3 decimals): {round(std_auc, 3)}")

Mean AUC: 0.7317
Standard Deviation: 0.0069 | Round (3 decimals): 0.007


In [21]:
# Question 6: Hyperparameter Tuning C

c_params = [0.000001, 0.001, 1]

for c in c_params:
    scores_c = []
    
    for train_idx, val_idx in kfold.split(df_full_train):
        df_tr = df_full_train.iloc[train_idx]
        df_va = df_full_train.iloc[val_idx]
        
        dv, clf = train_model(df_tr, y_full[train_idx], C=c)
        y_pred = predict_model(df_va, dv, clf)
        
        scores_c.append(roc_auc_score(y_full[val_idx], y_pred))
        
    mean_c = np.mean(scores_c)
    std_c = np.std(scores_c)
    print(f"C = {c:<8} -> Mean AUC: {mean_c:.3f} | Std: {std_c:.3f}")

C = 1e-06    -> Mean AUC: 0.617 | Std: 0.019
C = 0.001    -> Mean AUC: 0.749 | Std: 0.010
C = 1        -> Mean AUC: 0.732 | Std: 0.007
